<a href="https://colab.research.google.com/github/TerteryanTatev/Machine-Learning-Algorithms/blob/main/Naive-Bayes-Classifier/naive_bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import Counter

spam_docs = ["win money now", "win big prize"]
ham_docs = ["meeting schedule", "project meeting"]

def tokenize(text):
    return text.split()

vocab = set()
for doc in spam_docs + ham_docs:
    vocab.update(tokenize(doc))

V = len(vocab)

def train(docs):
    words = []
    for doc in docs:
        words.extend(tokenize(doc))
    return Counter(words), len(words)

spam_counts, spam_total = train(spam_docs)
ham_counts, ham_total = train(ham_docs)

def prob(word, counts, total):
    return (counts[word] + 1) / (total + V)

def classify(text):
    words = tokenize(text)

    p_spam = 0.5
    p_ham = 0.5

    for w in words:
        p_spam *= prob(w, spam_counts, spam_total)
        p_ham *= prob(w, ham_counts, ham_total)

    return p_spam, p_ham

print(classify("win meeting"))

(0.007653061224489795, 0.010416666666666666)


In [ ]:

spam_docs = [
    "win money now",
    "win big prize"
]

ham_docs = [
    "meeting schedule",
    "project meeting"
]

test_doc = "win meeting"


vocab = set()

for doc in spam_docs + ham_docs:
    vocab.update(doc.split())

vocab = sorted(vocab)
V = len(vocab)

print("Vocabulary:", vocab)
print("V =", V)

def doc_to_binary(doc, vocab):
    words = set(doc.split())
    return {w: (1 if w in words else 0) for w in vocab}

def count_docs_with_word(docs, word):
    return sum(1 for doc in docs if word in doc.split())

N_spam = len(spam_docs)
N_ham = len(ham_docs)

P_spam = N_spam / (N_spam + N_ham)
P_ham = N_ham / (N_spam + N_ham)

def P_word_given_class(word, docs, N_docs):
    count = count_docs_with_word(docs, word)
    return (count + 1) / (N_docs + 2)

def classify(doc):
    words = doc.split()

    spam_score = P_spam
    for w in words:
        spam_score *= P_word_given_class(w, spam_docs, N_spam)

    ham_score = P_ham
    for w in words:
        ham_score *= P_word_given_class(w, ham_docs, N_ham)

    return spam_score, ham_score

spam_score, ham_score = classify(test_doc)

print("\nScores:")
print("Spam:", spam_score)
print("Ham:", ham_score)

if spam_score > ham_score:
    print("Result: spam")
elif ham_score > spam_score:
    print("Result: ham")
else:
    print("Result: equal")

Vocabulary: ['big', 'meeting', 'money', 'now', 'prize', 'project', 'schedule', 'win']
V = 8

Scores:
Spam: 0.09375
Ham: 0.09375
Result: equal


In [ ]:
import numpy as np

docs = [
    ("win money now", "spam"),
    ("win big prize", "spam"),
    ("meeting schedule", "ham"),
    ("project meeting", "ham")
]

def extract_features(text):
    words = text.split()
    x1 = len(words)
    x2 = np.mean([len(w) for w in words])
    return np.array([x1, x2])

X = np.array([extract_features(t) for t, _ in docs])
y = np.array([label for _, label in docs])

spam_X = X[y == "spam"]
ham_X = X[y == "ham"]

def stats(data):
    mean = np.mean(data, axis=0)
    var = np.var(data, axis=0)
    return mean, var

spam_mean, spam_var = stats(spam_X)
ham_mean, ham_var = stats(ham_X)

P_spam = len(spam_X) / len(X)
P_ham = len(ham_X) / len(X)

print("========== TRAINING STATS ==========")
print("Spam mean:", spam_mean)
print("Spam var :", spam_var)
print("Ham mean :", ham_mean)
print("Ham var  :", ham_var)
print("P(spam)  :", P_spam)
print("P(ham)   :", P_ham)

def gaussian(x, mean, var):
    eps = 1e-6
    var = var + eps
    return (1 / np.sqrt(2 * np.pi * var)) * np.exp(-((x - mean) ** 2) / (2 * var))


def classify(text):
    x = extract_features(text)

    print("\n========== TEST SAMPLE ==========")
    print("Text:", text)
    print("Features:", x)

    print("\n--- SPAM CALCULATION ---")
    spam_score = P_spam
    print("Initial P(spam):", spam_score)

    for i in range(len(x)):
        g = gaussian(x[i], spam_mean[i], spam_var[i])
        print(f"Feature {i+1}: x={x[i]}, mean={spam_mean[i]}, var={spam_var[i]}, gaussian={g}")
        spam_score *= g
        print("Updated spam_score:", spam_score)

    print("\n--- HAM CALCULATION ---")
    ham_score = P_ham
    print("Initial P(ham):", ham_score)

    for i in range(len(x)):
        g = gaussian(x[i], ham_mean[i], ham_var[i])
        print(f"Feature {i+1}: x={x[i]}, mean={ham_mean[i]}, var={ham_var[i]}, gaussian={g}")
        ham_score *= g
        print("Updated ham_score:", ham_score)

    print("\n========== FINAL RESULT ==========")
    print("Spam score:", spam_score)
    print("Ham score :", ham_score)

    if spam_score > ham_score:
        print("RESULT: SPAM")
    else:
        print("RESULT: HAM")

classify("win meeting")

========== TRAINING STATS ==========
Spam mean: [3.         3.66666667]
Spam var : [0. 0.]
Ham mean : [2.   7.25]
Ham var  : [0.     0.0625]
P(spam)  : 0.5
P(ham)   : 0.5

========== TEST SAMPLE ==========
Text: win meeting
Features: [2. 5.]

--- SPAM CALCULATION ---
Initial P(spam): 0.5
Feature 1: x=2.0, mean=3.0, var=0.0, gaussian=0.0
Updated spam_score: 0.0
Feature 2: x=5.0, mean=3.6666666666666665, var=0.0, gaussian=0.0
Updated spam_score: 0.0

--- HAM CALCULATION ---
Initial P(ham): 0.5
Feature 1: x=2.0, mean=2.0, var=0.0, gaussian=398.94228040143264
Updated ham_score: 199.47114020071632
Feature 2: x=5.0, mean=7.25, var=0.0625, gaussian=4.114541850605076e-18
Updated ham_score: 8.207323543437599e-16

========== FINAL RESULT ==========
Spam score: 0.0
Ham score : 8.207323543437599e-16
RESULT: HAM
